# Project 2 · Step 4: recurrent encoder (Kazik)

Pipeline for step 4: data, pretrained embeddings, tokenisation, padded batches, the shared folds, a training loop, an LSTM skeleton (section 7, **yours to extend**), a hyperparameter grid, an overfitting check and the single final test evaluation.

Everything uses the team's shared protocol, so the numbers compare directly with the baselines in `shared/results/`:
- `emb-ft-crawl-mean/logreg`: mean-pooled fastText + logistic regression, **28.5%** balanced accuracy;
- `emb-ft-crawl-mean-promptcentred/logreg`: the same with each prompt's mean subtracted, **40.6%**.

Chance is 5.6% (18 classes).

## 1. Setup

In [ ]:
import copy
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "part2" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

from part2 import wordvecs as WV
from shared import eval_harness as H
from shared.data import load_data

DEVICE = "cpu"  # "mps" also works on a Mac, but CPU runs are reproducible and LSTMs this small aren't much faster on MPS
H.set_seed()

## 2. Data

`D.train` is our training split (2,159 passages). `D.test` is the held-out split: **don't touch it** until the final evaluation. Labels are integers `0..17` and `CLASSES[i]` is the model name. `groups` is the prompt id, which the folds use so that no prompt appears on both sides of a split.

In [ ]:
D = load_data()
texts = D.train["text"].tolist()
y = D.y  # np.ndarray of ints, aligned with texts
groups = D.groups  # prompt id per text
CLASSES = D.classes
N_CLASSES = len(CLASSES)

print(
    f"train {len(texts)} texts, test (held out) {len(D.test)}, {N_CLASSES} classes, {len(set(groups))} prompts"
)
pd.Series(y).map(dict(enumerate(CLASSES))).value_counts().sort_index().to_frame("n").T

## 3. Embeddings

`WV.load_subset(name)` returns `(tok, emb)`:
- `tok` is the tokeniser that matches that embedding's vocabulary. It is cased, keeps punctuation as tokens, and splits `don't` the way the embedding was trained.
- `emb.vectors` is a `(V, 300)` numpy array. **Row 0 is `<pad>`** (all zeros) and **row 1 is `<unk>`**. The table holds only the words that occur in our texts (train and Kaggle test), which keeps it small.

Useful for later experiments:
- `emb.random_like(seed)` gives the same vocabulary with random vectors (pretrained vs random initialisation);
- `emb.pca(k)` gives the same vocabulary in k dimensions.

The main embedding is `ft-crawl` (fastText Common Crawl). The others load only if their subset has been built (`python -m part2.prepare_embeddings <name>`).

**`emb_matrix(name)` standardises each of the 300 dimensions** (mean 0, std 1 over the vocabulary; the `<pad>` row stays 0), the same scaling the logistic-regression baseline applies with `StandardScaler`. The raw vectors are tiny, and on them the models barely train: on the same 5 folds, an LSTM (100 units, 1 layer) went from about 16% to 20.5% balanced accuracy with standardisation. `emb_matrix(name, standardise=False)` gives the raw vectors.

In [ ]:
PAD_IDX, UNK_IDX = 0, 1

EMB = {}
for name in ["ft-crawl", "glove-840b", "ft-wiki", "w2v-gnews"]:
    try:
        EMB[name] = WV.load_subset(name)
    except FileNotFoundError as e:
        print(f"skip {name}: {e}")

for name, (tok, emb) in EMB.items():
    assert emb.vocab.index("<pad>") == PAD_IDX and emb.vocab.index("<unk>") == UNK_IDX
    print(f"{name:11s} {emb.vectors.shape[0]:,} words x {emb.vectors.shape[1]}d")

EMB_NAME = "ft-crawl"  # the one to use; change here to switch
tok, emb = EMB[EMB_NAME]


def emb_matrix(name, standardise=True):
    """(V, 300) float32 tensor for embedding `name`; its rows match the ids in IDS[name].
    standardise: z-score each dimension over the vocabulary rows, as the baseline's StandardScaler does;
    the <pad> row stays 0. Raw vectors are tiny (|component| ~0.05) and the models train poorly on them.
    """
    v = EMB[name][1].vectors
    if standardise:
        words = v[2:]  # all rows except <pad> and <unk>
        v = (v - words.mean(0)) / words.std(0)  # new array: emb.vectors is not modified
        v[PAD_IDX] = 0.0
    return torch.tensor(v, dtype=torch.float32)


E = emb_matrix(EMB_NAME)
E.shape

## 4. Tokenise once

Every text becomes a list of token ids, once per embedding (each embedding has its own tokeniser and vocabulary). The ids are cached in `IDS[name]` for the train split and `IDS_TEST[name]` for the held-out test split. **`IDS_TEST` is only used in section 10 (final test).**

In [ ]:
IDS = {name: [t.ids(text) for text in texts] for name, (t, _) in EMB.items()}
IDS_TEST = {name: [t.ids(text) for text in D.test["text"]] for name, (t, _) in EMB.items()}
y_test = D.y_test

ex = texts[0]
print(ex[:300], "...\n")
print("tokens:", tok.tokens(ex)[:25])
print("ids:   ", IDS[EMB_NAME][0][:25])

lengths = pd.Series([len(s) for s in IDS[EMB_NAME]])
unk_share = np.mean([i == UNK_IDX for s in IDS[EMB_NAME] for i in s])
print(
    f"\ntokens per text: median {lengths.median():.0f}, 95th pct {lengths.quantile(.95):.0f}, max {lengths.max()}"
)
print(f"<unk> share of tokens: {unk_share:.4%}")
assert lengths.min() > 0

## 5. Dataset and batches

`collate` pads each batch to its longest text with `PAD_IDX` and also returns the true lengths, which `pack_padded_sequence` needs, so the LSTM never reads padding.

Each batch is `x` (B, T) long, `lengths` (B,) long and `y` (B,) long. `make_loader(idx, emb_name)` indexes the train split; `test=True` indexes the test split instead.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, seqs, labels):
        self.seqs = [torch.tensor(s, dtype=torch.long) for s in seqs]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.seqs[i], self.labels[i]


def collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    x = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX)
    return x, lengths, torch.stack(labels)


def make_loader(idx, emb_name=EMB_NAME, batch_size=32, shuffle=False, test=False):
    """DataLoader over the texts at positions `idx` (e.g. one fold), tokenised for `emb_name`.
    Positions refer to the train split, or to the test split if `test=True`."""
    ids, labels = (IDS_TEST, y_test) if test else (IDS, y)
    ds = TextDataset([ids[emb_name][i] for i in idx], labels[idx])
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)


x, lens, yb = next(iter(make_loader(np.arange(64), shuffle=True)))
print("x", tuple(x.shape), x.dtype, "| lengths", tuple(lens.shape), "| y", tuple(yb.shape))

## 6. Folds

These are the team's shared folds (`shared/folds.json`): 5 folds grouped by prompt, with 10 repeats available. Neural models are slow, so we use 1 repeat (5 folds) by default.

Inside each outer training fold, `H.inner_split` holds out 20% of the prompts as an **inner validation set** for early stopping and model selection. The outer test fold is only used for the reported score.

In [ ]:
FOLDS = H.load_folds(
    D.train["id"].tolist(), repeats=1
)  # list of (repeat, fold, train_idx, test_idx)

r, k, tr, te = FOLDS[0]
itr, iva = H.inner_split(tr, groups, seed=H.SEED + r)
print(f"{len(FOLDS)} folds; fold 0: inner train {len(itr)}, inner val {len(iva)}, test {len(te)}")
P = lambda idx: set(groups[idx])
assert (
    not (P(itr) & P(iva)) and not (P(itr) & P(te)) and not (P(iva) & P(te))
), "prompts leak between splits"

## 7. The model

Frozen pretrained embeddings → LSTM (`num_layers` stacked layers of `hidden_size` units) → **mean over the top layer's outputs at every real token** → dropout → one linear layer giving 18 logits (no softmax: the loss applies it).

- **Mean pooling:** the text is represented by the average of the LSTM's output at every token, not only its final state. Information from the whole text counts equally, as in the mean-pooled embedding baseline. `pack_padded_sequence` / `pad_packed_sequence` keep padding out: padded positions come back as zeros and the sum is divided by each text's real length.
- **Dropout between layers** (`dropout`, a required argument searched in the grid, active only in training): between stacked LSTM layers (`nn.LSTM(dropout=...)`, which only applies when `num_layers > 1`) and between the LSTM and the linear layer, so 1-layer models are regularised too.

The contract the training loop needs: `model(x, lengths)` returns logits of shape (B, N_CLASSES), and a no-argument `make_model()` builds a fresh model (one per fold).

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, E, hidden_size, num_layers, dropout, n_classes=N_CLASSES, pad_idx=PAD_IDX):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(E, freeze=True, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            E.shape[1],
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )  # between stacked LSTM layers
        self.drop = nn.Dropout(dropout)  # between the LSTM and the linear layer
        self.out = nn.Linear(hidden_size, n_classes)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            self.emb(x), lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        outputs, _ = self.lstm(packed)
        outputs, _ = pad_packed_sequence(
            outputs, batch_first=True
        )  # (B, T, hidden_size), zeros after each text ends
        pooled = outputs.sum(1) / lengths.to(outputs.device).unsqueeze(
            1
        )  # mean over real tokens -> (B, hidden_size)
        return self.out(self.drop(pooled))  # (B, N_CLASSES)

## 8. Training and evaluation

`train_fold` trains one model with **Adam** and an **exponentially decaying learning rate** (`lr` × `lr_decay`^epoch):
1. train on the inner train split;
2. after each epoch, compute the loss on the inner validation split, keep the best weights, and stop after `patience` epochs without improvement;
3. return the model with its best weights.

`run_cv` does this for each outer fold and scores the model on the outer test fold (balanced accuracy, accuracy, macro F1). `verbose` sets how much progress is printed: `0` only the final line, `1` (default) one line per fold, `2` also one line per epoch (learning rate, train and validation loss, `*` when the validation loss improved). Each fold row also stores the train and validation loss and the learning rate per epoch (`history`), used in section 9b to look at overfitting.

`run_cv` returns a dict in the team's result format, so `H.save_result(res)` writes it to `shared/results/` and `H.compare(res, baseline)` gives the Bayesian comparison (ROPE 1 pp).

In [ ]:
def predict_logprobs(model, loader, device=DEVICE):
    model.eval()
    out = []
    with torch.no_grad():
        for x, lens, _ in loader:
            out.append(torch.log_softmax(model(x.to(device), lens), -1).cpu())
    return torch.cat(out).numpy()


def train_fold(
    make_model,
    tr,
    va,
    emb_name=EMB_NAME,
    lr=1e-3,
    lr_decay=0.9,
    weight_decay=0.0,
    epochs=30,
    patience=5,
    batch_size=32,
    device=DEVICE,
    verbose=False,
):
    """Train on train-split positions `tr`, early-stop on the loss of `va`. Returns (model, best_epoch, history).
    verbose=True prints one line per epoch."""
    model = make_model().to(device)
    opt = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay
    )
    sched = torch.optim.lr_scheduler.ExponentialLR(
        opt, gamma=lr_decay
    )  # lr is multiplied by lr_decay each epoch
    loss_fn = nn.CrossEntropyLoss()
    train_dl = make_loader(tr, emb_name, batch_size, shuffle=True)
    val_dl = make_loader(va, emb_name, 256)

    best, best_state, best_epoch, history = float("inf"), None, -1, []
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for x, lens, yb in train_dl:
            opt.zero_grad()
            loss = loss_fn(model(x.to(device), lens), yb.to(device))
            loss.backward()
            opt.step()
            total += loss.item() * len(yb)
        val_logp = predict_logprobs(model, val_dl, device)
        val_loss = float(-val_logp[np.arange(len(va)), y[va]].mean())
        history.append(
            {
                "epoch": epoch,
                "lr": sched.get_last_lr()[0],
                "train_loss": total / len(tr),
                "val_loss": val_loss,
            }
        )
        sched.step()
        improved = val_loss < best
        if improved:
            best, best_state, best_epoch = val_loss, copy.deepcopy(model.state_dict()), epoch
        if verbose:
            h = history[-1]
            print(
                f"    epoch {epoch:2d}  lr {h['lr']:.2e}  train {h['train_loss']:.4f}  val {val_loss:.4f}"
                f"{'  *' if improved else ''}  ({time.time() - t0:.0f}s)",
                flush=True,
            )
        if not improved and epoch - best_epoch >= patience:
            if verbose:
                print(
                    f"    early stop: no val improvement for {patience} epochs, keeping epoch {best_epoch}"
                )
            break

    model.load_state_dict(best_state)
    return model, best_epoch, history


def run_cv(make_model, name, emb_name=EMB_NAME, folds=None, verbose=1, **train_kw):
    """CV on the shared folds (early stopping on an inner split of each training fold).
    `name` follows the team convention, e.g. "ft-crawl-lstm-h100-l2/linear".
    verbose: 0 = summary line only, 1 = one line per fold, 2 = also one line per epoch."""
    folds = FOLDS if folds is None else folds
    rows, oof_pred, oof_proba = [], np.full(len(y), -1), np.zeros((len(y), N_CLASSES))
    t0 = time.time()
    for r, k, tr, te in folds:
        H.set_seed(H.SEED + r)
        itr, iva = H.inner_split(tr, groups, seed=H.SEED + r)
        if verbose >= 2:
            print(f"  fold {k}: train {len(itr)}, val {len(iva)}, test {len(te)}", flush=True)
        model, best_epoch, history = train_fold(
            make_model, itr, iva, emb_name, verbose=verbose >= 2, **train_kw
        )
        logp = predict_logprobs(model, make_loader(te, emb_name, 256))
        pred = logp.argmax(1)
        rows.append(
            {
                "repeat": r,
                "fold": k,
                **H.scores(y[te], pred),
                "best_epoch": best_epoch,
                "history": history,
                "n_test": len(te),
                "n_total": len(tr) + len(te),
            }
        )
        if r == folds[0][0]:
            oof_pred[te], oof_proba[te] = pred, np.exp(logp)
        if verbose:
            print(
                f"  fold {k}: bal.acc {rows[-1]['balanced_accuracy']:.4f}  (best epoch {best_epoch}, "
                f"{time.time() - t0:.0f}s)",
                flush=True,
            )
    res = {
        "name": name,
        "model": "torch",
        "folds": rows,
        "oof_pred": oof_pred,
        "oof_proba": oof_proba,
        "seconds": time.time() - t0,
        "runs": len({f[0] for f in folds}),
        "config": {"emb": emb_name, **train_kw},
    }
    ba = H.fold_scores(res)
    print(f"{name}: bal.acc {ba.mean():.4f} ± {ba.std():.4f} over {len(rows)} folds")
    return res

### Smoke test

Checks the output shape, then runs a small LSTM for 2 epochs on one fold. It only shows the pipeline runs end to end; it is **not** an experiment, and nothing is saved.

In [ ]:
_m = LSTMClassifier(E, hidden_size=50, num_layers=2, dropout=0.3)
_x, _lens, _ = next(iter(make_loader(np.arange(8))))
assert _m(_x, _lens).shape == (8, N_CLASSES)

_smoke = run_cv(
    lambda: LSTMClassifier(E, 50, 1, dropout=0.3), "smoke", folds=FOLDS[:1], epochs=2, verbose=2
)

## 9. Hyperparameter search

`grid_search(emb_name)` runs the CV above (5 folds × 1 repeat over the whole train split) for every combination of
- hidden units: 64, 128, 256;
- LSTM layers: 1, 2 (3+ layers did worse in earlier checks);
- dropout: 0.2, 0.5;

that is 12 configs, with Adam and the decaying learning rate (`lr=1e-3`, `lr_decay=0.9`) fixed. Override any of them, e.g. `grid_search(..., dropout=(0.3,), lr=...)`.

It returns a summary table sorted by mean balanced accuracy and a dict `{(hidden, layers, dropout): result}` with the full results. The table is also written to `part2/lstm_grid_<emb>.csv`, so it survives a kernel restart.

Each fold's score comes from a fold the model never saw, but picking the best of 12 configs on those same scores makes the winner's CV number slightly optimistic. The test split (section 10) gives the unbiased number.

In [ ]:
HIDDEN = (64, 128, 256)
LAYERS = (1, 2)
DROPOUT = (0.2, 0.5)


def grid_search(
    emb_name,
    hidden=HIDDEN,
    layers=LAYERS,
    dropout=DROPOUT,
    folds=None,
    save_csv=True,
    verbose=1,
    **train_kw,
):
    """verbose: 0 = one line per config, 1 = also one line per fold, 2 = also one line per epoch."""
    E_ = emb_matrix(emb_name)
    configs = [(h, l, d) for h in hidden for l in layers for d in dropout]
    rows, results, t0 = [], {}, time.time()
    for i, (h, l, d) in enumerate(configs, 1):
        print(
            f"[{i}/{len(configs)}] {emb_name}: hidden {h}, layers {l}, dropout {d} ({(time.time() - t0) / 60:.1f} min elapsed)",
            flush=True,
        )
        name = f"{emb_name}-lstm-h{h}-l{l}-d{d}/linear"
        res = run_cv(
            lambda h=h, l=l, d=d: LSTMClassifier(E_, h, l, dropout=d),
            name,
            emb_name,
            folds=folds,
            verbose=verbose,
            **train_kw,
        )
        res["config"].update(hidden=h, layers=l, dropout=d, pooling="mean")
        results[(h, l, d)] = res
        f = pd.DataFrame(res["folds"])
        rows.append(
            {
                "hidden": h,
                "layers": l,
                "dropout": d,
                "bal_acc": f["balanced_accuracy"].mean(),
                "bal_acc_std": f["balanced_accuracy"].std(ddof=0),
                "accuracy": f["accuracy"].mean(),
                "macro_f1": f["macro_f1"].mean(),
                "best_epoch": f["best_epoch"].mean(),
                "minutes": res["seconds"] / 60,
            }
        )
    summary = pd.DataFrame(rows).sort_values("bal_acc", ascending=False).reset_index(drop=True)
    if save_csv:
        summary.to_csv(ROOT / "part2" / f"lstm_grid_{emb_name}.csv", index=False)
    return summary, results

### 9b. Overfitting

`plot_curves(res)` draws, for each fold, the training loss and the inner-validation loss per epoch; the dashed line is the epoch whose weights were kept. Overfitting shows up as the training loss still falling while the validation loss turns back up. The table below the plot gives both losses at the kept epoch and their gap.

In [ ]:
TRAIN_C, VAL_C, MUTED = "#2a78d6", "#eb6834", "#8a8a85"  # categorical slots 1-2; neutral for guides


def plot_curves(res):
    folds = res["folds"]
    fig, axes = plt.subplots(
        1, len(folds), figsize=(3.2 * len(folds), 3), sharey=True, squeeze=False
    )
    for ax, f in zip(axes[0], folds):
        h = pd.DataFrame(f["history"])
        ax.plot(h["epoch"], h["train_loss"], color=TRAIN_C, lw=2, label="train")
        ax.plot(h["epoch"], h["val_loss"], color=VAL_C, lw=2, label="inner validation")
        ax.axvline(f["best_epoch"], color=MUTED, lw=1, ls="--")
        ax.set_title(f"fold {f['fold']} · bal.acc {f['balanced_accuracy']:.3f}", fontsize=10)
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(axis="y", color="#e6e6e3", lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0][0].set_ylabel("cross-entropy loss")
    axes[0][0].legend(frameon=False, fontsize=9)
    fig.suptitle(res["name"], fontsize=11)
    fig.tight_layout()
    plt.show()

    at_best = pd.DataFrame(
        [
            {
                "fold": f["fold"],
                "best_epoch": f["best_epoch"],
                "train_loss": f["history"][f["best_epoch"]]["train_loss"],
                "val_loss": f["history"][f["best_epoch"]]["val_loss"],
                "outer_bal_acc": f["balanced_accuracy"],
            }
            for f in folds
        ]
    )
    at_best["gap"] = at_best["val_loss"] - at_best["train_loss"]
    return at_best.round(3)


plot_curves(_smoke)  # smoke-test run, just to show the plot; replace with a real result

## 10. Final test (run once)

`final_test` is the single evaluation on the held-out test split, following the team's `H.final_test` convention: hold out 20% of the train split's prompts for early stopping, train on the rest with the chosen config, and score the test split once. **Only call it after the config is chosen by CV.**

In [ ]:
def final_test(emb_name, hidden, layers, dropout, verbose=True, **train_kw):
    H.set_seed(H.SEED)
    itr, iva = H.inner_split(np.arange(len(y)), groups, seed=H.SEED)
    E_ = emb_matrix(emb_name)
    model, best_epoch, history = train_fold(
        lambda: LSTMClassifier(E_, hidden, layers, dropout),
        itr,
        iva,
        emb_name,
        verbose=verbose,
        **train_kw,
    )
    pred = predict_logprobs(
        model, make_loader(np.arange(len(y_test)), emb_name, 256, test=True)
    ).argmax(1)
    return {
        "name": f"{emb_name}-lstm-h{hidden}-l{layers}-d{dropout}/linear",
        "emb": emb_name,
        "hidden": hidden,
        "layers": layers,
        "dropout": dropout,
        **H.scores(y_test, pred),
        "best_epoch": best_epoch,
        "history": history,
        "pred": pred.tolist(),
    }

## 11. Full pipeline per embedding

`run_pipeline(emb_name)`:
1. runs the 12-config grid (section 9) and shows the table;
2. picks the config with the highest mean balanced accuracy and saves its CV result to `shared/results/<emb>-lstm-h<h>-l<l>-d<d>__linear.json` (the team format, so it can be compared with the baselines);
3. plots its loss curves (section 9b);
4. runs the final test (section 10) and writes the scores to `part2/lstm_test_<emb>.json`.

`hidden`, `layers` and `dropout` (tuples of values to search) go to the grid; `**train_kw` (e.g. `weight_decay=1e-4`) are passed on to the grid and the final test. `verbose` works as in `grid_search` (`2` prints every epoch). Compare with a baseline afterwards:
```python
R = H.load_results()
H.compare(R["emb-ft-crawl-mean/logreg"], ft_results[(h, l, d)])   # P(A>B), P(ROPE), P(B>A)
```

In [ ]:
import json


def run_pipeline(
    emb_name, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT, folds=None, verbose=1, **train_kw
):
    """Grid arguments (hidden, layers, dropout, folds) go to the grid only; **train_kw (lr, epochs, ...) to both."""
    summary, results = grid_search(
        emb_name, hidden, layers, dropout, folds=folds, verbose=verbose, **train_kw
    )
    display(summary.round(4))

    h, l, d = (
        int(summary.loc[0, "hidden"]),
        int(summary.loc[0, "layers"]),
        float(summary.loc[0, "dropout"]),
    )
    best = results[(h, l, d)]
    print(f"chosen: hidden {h}, layers {l}, dropout {d} -> saved {H.save_result(best)}")
    display(plot_curves(best))

    print("final test: training on the whole train split")
    test = final_test(emb_name, h, l, d, verbose=verbose >= 1, **train_kw)
    (ROOT / "part2" / f"lstm_test_{emb_name}.json").write_text(json.dumps(test))
    print(
        f"TEST {test['name']}: bal.acc {test['balanced_accuracy']:.4f}, acc {test['accuracy']:.4f}, "
        f"macro F1 {test['macro_f1']:.4f}"
    )
    return summary, results, test

**Each of the two cells below takes roughly 1–2 hours on CPU (9 configs × 5 folds × up to 30 epochs) and evaluates on the test split.** Run them when the model in section 7 is final.

In [ ]:
ft_summary, ft_results, ft_test = run_pipeline("ft-crawl")

In [ ]:
glove_summary, glove_results, glove_test = run_pipeline("glove-840b")